# Week 11 — Network Architecture Design

Every model so far was a stack: one input, layers in a line, one output.
This notebook is about *wiring* — building the network as a graph around
mixed tabular data, using the Keras Functional API.

The case study: predicting survival on the Titanic passenger manifest
(loaded from seaborn's public dataset collection). We climb a ladder of
five models:

1. One input, Sequential vs Functional (same model, two APIs)
2. Bucketing a numeric input (Discretization)
3. Multiple named inputs (numbers + categories, one-hot)
4. Embeddings for categorical columns
5. Multiple outputs (two heads, per-head losses)

Baselines come first, as always. Run every cell; change things and rerun.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

df = sns.load_dataset("titanic")[["survived", "age", "fare", "sex", "pclass"]].copy()
print(df.shape)
print(df.isna().sum())
df.head()

## SETUP: missing values and a train/dev split

`age` is missing for about a fifth of passengers. We impute the median —
an assumption, and one you should be able to defend. Then a shuffled
80/20 train/dev split. The dev set is where every model in this notebook
is judged.

In [ ]:
df["age"] = df["age"].fillna(df["age"].median())

idx = np.random.permutation(len(df))
cut = int(0.8 * len(df))
train, dev = df.iloc[idx[:cut]].copy(), df.iloc[idx[cut:]].copy()

y_train = train["survived"].values.astype("float32")
y_dev = dev["survived"].values.astype("float32")
print(len(train), "train /", len(dev), "dev")
print("survival rate (train): %.3f" % y_train.mean())

## PART 1: BASELINES

A network that cannot beat one honest rule is decoration. Two rules:

- **Majority class**: predict nobody survived.
- **One-rule**: predict survived if and only if `sex == "female"`.

In [ ]:
maj = np.zeros_like(y_dev)
rule = (dev["sex"] == "female").values.astype("float32")

print("majority-class accuracy: %.3f" % (maj == y_dev).mean())
print("one-rule (sex) accuracy: %.3f" % (rule == y_dev).mean())

## PART 2: SEQUENTIAL VS FUNCTIONAL — same model, two APIs

One input (`age`), one sigmoid unit: logistic regression. First as a
`Sequential` stack, then as a Functional graph. Identical parameters,
identical model — the Functional version just makes the graph explicit:
layers are *called on tensors*, and the model is whatever runs from
inputs to outputs.

In [ ]:
x_age_train = train[["age"]].values.astype("float32")
x_age_dev = dev[["age"]].values.astype("float32")

norm_age = layers.Normalization()
norm_age.adapt(x_age_train)

seq = keras.Sequential([
    keras.Input(shape=(1,), name="age"),
    norm_age,
    layers.Dense(1, activation="sigmoid"),
])

inp = keras.Input(shape=(1,), name="age")
out = layers.Dense(1, activation="sigmoid")(norm_age(inp))
fun = keras.Model(inputs=inp, outputs=out)

for m in (seq, fun):
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(x_age_train, y_train, epochs=20, verbose=0)

print("sequential:", dict(zip(seq.metrics_names, seq.evaluate(x_age_dev, y_dev, verbose=0))))
print("functional:", dict(zip(fun.metrics_names, fun.evaluate(x_age_dev, y_dev, verbose=0))))
fun.summary()

Age alone barely moves past the majority baseline — no encoding of a
weak feature will fix that. But notice what the Functional version bought
us: `inp` and `out` are tensors we can branch, merge, and reuse. The rest
of the notebook lives on that freedom.

## PART 3: BUCKETING A NUMBER — Discretization

Age's effect on survival is not smooth: children were prioritized,
young adults were crew-aged, the elderly fared differently. A
`Discretization` layer turns the number into a handful of buckets so the
model can learn a separate effect per range — zero trainable parameters,
just a different front door.

In [ ]:
buckets = layers.Discretization(bin_boundaries=[12.0, 25.0, 40.0, 60.0],
                                output_mode="one_hot")

inp = keras.Input(shape=(1,), name="age")
out = layers.Dense(1, activation="sigmoid")(buckets(inp))
bucketed = keras.Model(inp, out)
bucketed.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
bucketed.fit(x_age_train, y_train, epochs=20, verbose=0)
print("bucketed age:", dict(zip(bucketed.metrics_names, bucketed.evaluate(x_age_dev, y_dev, verbose=0))))
bucketed.summary()

## PART 4: MULTIPLE NAMED INPUTS

Now the real wiring. Four features, each with its own path:

- `age`, `fare` — numeric, normalized
- `sex` — string category, StringLookup then one-hot
- `pclass` — integer category, IntegerLookup then one-hot

Each path is a named `Input`; `Concatenate` merges them into a shared
trunk. Data is fed as a dict keyed by input name — with four inputs,
names are not decoration, they are the contract.

In [ ]:
def feed(frame):
    return {
        "age": frame[["age"]].values.astype("float32"),
        "fare": frame[["fare"]].values.astype("float32"),
        "sex": frame["sex"].values,
        "pclass": frame["pclass"].values,
    }

norm_age = layers.Normalization(); norm_age.adapt(feed(train)["age"])
norm_fare = layers.Normalization(); norm_fare.adapt(feed(train)["fare"])
lookup_sex = layers.StringLookup(vocabulary=["male", "female"], output_mode="one_hot")
lookup_cls = layers.IntegerLookup(vocabulary=[1, 2, 3], output_mode="one_hot")

in_age = keras.Input(shape=(1,), name="age")
in_fare = keras.Input(shape=(1,), name="fare")
in_sex = keras.Input(shape=(1,), dtype="string", name="sex")
in_cls = keras.Input(shape=(1,), dtype="int64", name="pclass")

paths = [
    norm_age(in_age),
    norm_fare(in_fare),
    layers.Flatten()(lookup_sex(in_sex)),
    layers.Flatten()(lookup_cls(in_cls)),
]
trunk = layers.Dense(16, activation="relu")(layers.Concatenate()(paths))
out = layers.Dense(1, activation="sigmoid", name="survived")(trunk)

multi = keras.Model([in_age, in_fare, in_sex, in_cls], out)
multi.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
def tparams(m):
    return int(sum(np.prod(w.shape) for w in m.trainable_weights))

multi.fit(feed(train), y_train, epochs=30, verbose=0)
print("multi-input:", dict(zip(multi.metrics_names, multi.evaluate(feed(dev), y_dev, verbose=0))))
print("trainable parameters:", tparams(multi))

In [ ]:
# The graph you wired, drawn. (Falls back to summary() if pydot is absent.)
try:
    keras.utils.plot_model(multi, show_shapes=True, dpi=64)
except Exception as e:
    print("plot_model unavailable here (%s) - summary instead:" % type(e).__name__)
    multi.summary()

This is the first model that clearly beats the one-rule baseline — and
the first whose parameter count is worth reading. Do the arithmetic by
hand once: concat width, times trunk units, plus biases. The bill should
match the trainable-parameter count exactly. (`count_params()` also counts
the Normalization layers’ frozen mean/variance state — sum
`trainable_weights` shapes instead.)

## PART 5: EMBEDDINGS FOR CATEGORICAL COLUMNS

One-hot spends width and no parameters. An `Embedding` spends parameters
to buy geometry: each category gets a learned vector, and categories the
task treats alike drift together — module 9's idea, pointed at a tabular
column. With 2- and 3-category features the payoff is small; with
thousands of categories (zip codes, product ids) it is the only option
that scales.

In [ ]:
lk_sex = layers.StringLookup(vocabulary=["male", "female"])
lk_cls = layers.IntegerLookup(vocabulary=[1, 2, 3])

in_age = keras.Input(shape=(1,), name="age")
in_fare = keras.Input(shape=(1,), name="fare")
in_sex = keras.Input(shape=(1,), dtype="string", name="sex")
in_cls = keras.Input(shape=(1,), dtype="int64", name="pclass")

emb_sex = layers.Embedding(input_dim=3, output_dim=4)   # vocab + OOV token
emb_cls = layers.Embedding(input_dim=4, output_dim=4)

paths = [
    norm_age(in_age),
    norm_fare(in_fare),
    layers.Flatten()(emb_sex(lk_sex(in_sex))),
    layers.Flatten()(emb_cls(lk_cls(in_cls))),
]
trunk = layers.Dense(16, activation="relu")(layers.Concatenate()(paths))
out = layers.Dense(1, activation="sigmoid", name="survived")(trunk)

embedded = keras.Model([in_age, in_fare, in_sex, in_cls], out)
embedded.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
embedded.fit(feed(train), y_train, epochs=30, verbose=0)
print("embedded:", dict(zip(embedded.metrics_names, embedded.evaluate(feed(dev), y_dev, verbose=0))))
print("trainable parameters:", tparams(embedded))

# What did pclass learn? Three 4-d vectors (plus the OOV row).
w = emb_cls.get_weights()[0]
for i, cls_name in enumerate(["OOV", "1st", "2nd", "3rd"]):
    print(cls_name, np.round(w[i], 3))
print("distance 1st-2nd: %.3f" % np.linalg.norm(w[1] - w[2]))
print("distance 1st-3rd: %.3f" % np.linalg.norm(w[1] - w[3]))

## PART 6: MULTIPLE OUTPUTS — two heads, one trunk

The Functional API's last trick: a model can predict more than one
thing. We add a second head that predicts the passenger's fare bracket
(cheap / mid / expensive, by training-set quantiles) from the same
trunk. Each head gets its own loss; training minimizes their weighted
sum. A well-chosen auxiliary head can regularize the trunk — it asks the
shared representation to be useful twice.

In [ ]:
q1, q2 = train["fare"].quantile([1/3, 2/3]).values
def fare_bracket(frame):
    f = frame["fare"].values
    return np.digitize(f, [q1, q2]).astype("int32")   # 0, 1, 2

yb_train, yb_dev = fare_bracket(train), fare_bracket(dev)

in_age = keras.Input(shape=(1,), name="age")
in_sex = keras.Input(shape=(1,), dtype="string", name="sex")
in_cls = keras.Input(shape=(1,), dtype="int64", name="pclass")

paths = [
    norm_age(in_age),
    layers.Flatten()(lookup_sex(in_sex)),
    layers.Flatten()(lookup_cls(in_cls)),
]
trunk = layers.Dense(16, activation="relu")(layers.Concatenate()(paths))
out_surv = layers.Dense(1, activation="sigmoid", name="survived")(trunk)
out_fare = layers.Dense(3, activation="softmax", name="fare_bracket")(trunk)

two_headed = keras.Model([in_age, in_sex, in_cls], [out_surv, out_fare])
two_headed.compile(
    optimizer="adam",
    loss={"survived": "binary_crossentropy",
          "fare_bracket": "sparse_categorical_crossentropy"},
    loss_weights={"survived": 1.0, "fare_bracket": 0.5},
    metrics={"survived": ["accuracy"], "fare_bracket": ["accuracy"]},
)
feed2_train = {k: v for k, v in feed(train).items() if k != "fare"}
feed2_dev = {k: v for k, v in feed(dev).items() if k != "fare"}
two_headed.fit(feed2_train, {"survived": y_train, "fare_bracket": yb_train},
               epochs=30, verbose=0)
res = two_headed.evaluate(feed2_dev, {"survived": y_dev, "fare_bracket": yb_dev}, verbose=0)
print(dict(zip(two_headed.metrics_names, np.round(res, 3))))
print("trainable parameters:", tparams(two_headed))

Note what we deliberately did NOT feed this model: `fare` is now a
*target*, so it left the inputs. Predicting a value you also feed in is
the most polite form of leakage.

## SUMMARY

- **Baselines first.** Majority class and one honest rule set the bar;
  wiring that cannot clear it is decoration.
- **Sequential is the special case.** The Functional API treats layers
  as functions on tensors; stacks, branches, merges, and multiple heads
  are all the same idea.
- **Every column gets a front door**: raw numbers, Discretization
  buckets when the effect is not smooth, one-hot for small categorical
  vocabularies, embeddings when you want geometry or the vocabulary is
  large.
- **Read the parameter count like a bill.** Every wiring choice prices
  in parameters; `count_params()` should never surprise you.
- **Multiple outputs share a trunk** and train on a weighted sum of
  per-head losses — and moving a column from input to target is a
  wiring decision with leakage consequences.

Next module: what a model trained on people learns about people —
fairness, measured with the tools you already have.